# Feature Engineering

This notebook prepares the serve-level dataset for modeling by creating new variables from match context, serve characteristics, opponent profile, and tactical combinations.

The goal is to build features that are available before the serve so they can later be used for point-outcome prediction and serve recommendation.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/table_tennis_serves.csv")

df.head()

In [ ]:
df["point_won"] = (df["point_outcome"] == "won").astype(int)

The target variable is `point_won`, where 1 means the server won the point and 0 means the server lost the point.

In [ ]:
leakage_features = ["return_type","return_quality","return_placement","rally_length","point_end_type","rally_type_achieved","chop_rally_outcome"]
leakage_features

These variables occur after the serve, so they should not be used in the main prediction model. Including them would leak future information into the model.

In [ ]:
df["score_margin"] = df["server_score"] - df["receiver_score"]
df["total_points_played_in_game"] = df["server_score"] + df["receiver_score"]
df["is_tied"] = (df["server_score"] == df["receiver_score"]).astype(int)
df["is_trailing"] = (df["server_score"] < df["receiver_score"]).astype(int)
df["is_leading"] = (df["server_score"] > df["receiver_score"]).astype(int)
df["is_late_game"] = (df["total_points_played_in_game"] >= 16).astype(int)
df["is_deuce_or_later"] = ((df["server_score"] >= 10) & (df["receiver_score"] >= 10)).astype(int)
df["is_game_point_for_server"] = ((df["server_score"] >= 10) & (df["server_score"] > df["receiver_score"])).astype(int)
df["is_game_point_against_server"] = ((df["receiver_score"] >= 10) & (df["receiver_score"] > df["server_score"])).astype(int)

In [ ]:
df["serve_spin_combo"] = df["serve_type"] + "_" + df["spin_type"]
df["serve_length_spin_combo"] = df["serve_length"] + "_" + df["spin_type"]
df["serve_placement_combo"] = df["serve_type"] + "_" + df["placement_zone"]
df["full_serve_combo"] = df["serve_type"] + "_" + df["spin_type"] + "_" + df["serve_length"] + "_" + df["placement_zone"]

These features capture tactical serve patterns. A serve is not just defined by one attribute; its value often depends on the combination of type, spin, length, and placement.

In [ ]:
df["is_heavy_spin"] = (df["spin_intensity"] >= 3).astype(int)
df["is_low_spin"] = (df["spin_intensity"] <= 1).astype(int)
df["spin_length_interaction"] = df["spin_intensity"].astype(str) + "_" + df["serve_length"]

In [ ]:
df["opponent_is_looper"] = (df["opponent_style"] == "looper").astype(int)
df["opponent_is_chopper"] = (df["opponent_style"] == "chopper").astype(int)
df["opponent_is_attacker"] = (df["opponent_style"] == "attacker").astype(int)

In [ ]:
combo_summary = (df.groupby("full_serve_combo").agg(combo_attempts=("point_won", "count"), combo_win_rate=("point_won", "mean")).reset_index())
df = df.merge(combo_summary, on="full_serve_combo", how="left")
df["combo_reliability"] = np.minimum(df["combo_attempts"] / 30, 1)

Serve combinations with very few attempts can have misleading win rates. The reliability score adjusts for sample size, giving more trust to serve patterns that appear more frequently in the dataset.

In [ ]:
df.to_csv("../data/processed/table_tennis_serves_features.csv", index=False)

This processed dataset will be used in the modeling notebook.